In [0]:
import joblib
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, StringType

MODEL_PATH = "/Volumes/aegis_fraud_workspace/finguard/finguard_volume/artifacts/models/isolation_forest_v1.pkl"

# Distributed batch scoring with Vectorized Pandas UDF
@F.pandas_udf(DoubleType())
def batch_anomaly_inference(amounts: pd.Series, ratios: pd.Series, hops: pd.Series) -> pd.Series:
    # Load model on executor
    model = joblib.load(MODEL_PATH)
    
    features = pd.DataFrame({
        "amount": amounts,
        "limit_ratio": ratios.fillna(1.0),
        "is_out_of_home": hops
    })
    
    # Negative decision scores indicate statistical outliers
    scores = model.decision_function(features)
    return pd.Series(scores)

# Automated SAR (Suspicious Activity Report) forensic summary
@F.udf(StringType())
def format_sar_investigative_narrative(txn_id, user_id, amt, limit, home, loc, score):
    flags = []
    if loc != home:
        flags.append(f"Geo mismatch: initiated at '{loc}' vs profile city '{home}'")
    if amt > limit:
        flags.append(f"Spending ceiling breach: ${amt:.2f} exceeds limit ${limit:.2f}")
    if score < -0.05:
        flags.append(f"High statistical anomaly index ({score:.4f})")
        
    summary = " | ".join(flags) if flags else "Behavioral spend anomaly detected."
    return f"AI TRIAGE: Flagged for {user_id}. {summary} -> Recommendation: Place authorization hold."

print("[SUCCESS] Scoring UDF and SAR narrative functions registered.")

[SUCCESS] Scoring UDF and SAR narrative functions registered.


In [0]:
from pyspark.sql import functions as F

chk_gold = "/Volumes/aegis_fraud_workspace/finguard/finguard_volume/_checkpoints/gold"

df_silver_stream = spark.readStream.table("aegis_fraud_workspace.finguard.silver_transactions")

# Run scoring and alert triage logic
df_scored_stream = (
    df_silver_stream
    .withColumn("anomaly_score", batch_anomaly_inference(F.col("amount"), F.col("limit_ratio"), F.col("is_out_of_home")))
    .withColumn("is_threat", F.when(F.col("anomaly_score") < -0.035, 1).otherwise(0))
    .withColumn(
        "sar_narrative",
        F.when(
            F.col("is_threat") == 1,
            format_sar_investigative_narrative(
                F.col("transaction_id"), F.col("user_id"), F.col("amount"),
                F.col("daily_limit"), F.col("home_city"), F.col("location"), F.col("anomaly_score")
            )
        ).otherwise(F.lit(None))
    )
)

# Microbatch handler to store threat records and print security triage logs
def process_gold_and_alerts(batch_df, batch_id):
    threats_df = batch_df.filter(F.col("is_threat") == 1)
    
    (
        threats_df.write
        .format("delta")
        .mode("append")
        .saveAsTable("aegis_fraud_workspace.finguard.gold_fraud_alerts")
    )
    
    incidents = threats_df.collect()
    if incidents:
        print(f"\n=======================================================")
        print(f"[ALERT DISPATCH - BATCH {batch_id}] {len(incidents)} SUSPICIOUS TRANSACTION(S)")
        print(f"=======================================================")
        for rec in incidents:
            print(f"• TXN: {rec['transaction_id']} | User: {rec['user_id']} | Amount: ${rec['amount']:.2f}")
            print(f"  Score: {rec['anomaly_score']:.4f}")
            print(f"  Notes: {rec['sar_narrative']}\n")
    else:
        print(f"[INFO - BATCH {batch_id}] No threats detected in this micro-batch.")

gold_writer = (
    df_scored_stream.writeStream
    .foreachBatch(process_gold_and_alerts)
    .option("checkpointLocation", chk_gold)
    .trigger(availableNow=True)
    .start()
)

gold_writer.awaitTermination()
print(f"[SUCCESS] Gold evaluation completed.")

[SUCCESS] Gold evaluation completed.


In [0]:
%sql
SELECT 
    transaction_id,
    user_id,
    amount,
    daily_limit,
    limit_ratio,
    home_city,
    location,
    round(anomaly_score, 4) AS anomaly_score,
    sar_narrative
FROM aegis_fraud_workspace.finguard.gold_fraud_alerts
ORDER BY anomaly_score ASC;

transaction_id,user_id,amount,daily_limit,limit_ratio,home_city,location,anomaly_score,sar_narrative
TXN_9208861,USR_010,7057.86,3500.0,2.0165,Chennai,Singapore,-0.1673,AI TRIAGE: Flagged for USR_010. Geo mismatch: initiated at 'Singapore' vs profile city 'Chennai' | Spending ceiling breach: $7057.86 exceeds limit $3500.00 | High statistical anomaly index (-0.1673) -> Recommendation: Place authorization hold.
TXN_4651113,USR_007,5524.05,6000.0,0.9207,Hyderabad,Dubai,-0.1673,AI TRIAGE: Flagged for USR_007. Geo mismatch: initiated at 'Dubai' vs profile city 'Hyderabad' | High statistical anomaly index (-0.1673) -> Recommendation: Place authorization hold.
TXN_3795603,USR_001,6262.18,5000.0,1.2524,Hyderabad,Dubai,-0.1673,AI TRIAGE: Flagged for USR_001. Geo mismatch: initiated at 'Dubai' vs profile city 'Hyderabad' | Spending ceiling breach: $6262.18 exceeds limit $5000.00 | High statistical anomaly index (-0.1673) -> Recommendation: Place authorization hold.
TXN_8838818,USR_007,4460.14,6000.0,0.7434,Hyderabad,London,-0.1673,AI TRIAGE: Flagged for USR_007. Geo mismatch: initiated at 'London' vs profile city 'Hyderabad' | High statistical anomaly index (-0.1673) -> Recommendation: Place authorization hold.
TXN_4673271,USR_003,5868.79,7500.0,0.7825,Mumbai,Singapore,-0.1673,AI TRIAGE: Flagged for USR_003. Geo mismatch: initiated at 'Singapore' vs profile city 'Mumbai' | High statistical anomaly index (-0.1673) -> Recommendation: Place authorization hold.
TXN_9010226,USR_008,4926.88,2000.0,2.4634,Pune,Singapore,-0.1673,AI TRIAGE: Flagged for USR_008. Geo mismatch: initiated at 'Singapore' vs profile city 'Pune' | Spending ceiling breach: $4926.88 exceeds limit $2000.00 | High statistical anomaly index (-0.1673) -> Recommendation: Place authorization hold.
TXN_6675895,USR_001,7481.58,5000.0,1.4963,Hyderabad,Dubai,-0.1673,AI TRIAGE: Flagged for USR_001. Geo mismatch: initiated at 'Dubai' vs profile city 'Hyderabad' | Spending ceiling breach: $7481.58 exceeds limit $5000.00 | High statistical anomaly index (-0.1673) -> Recommendation: Place authorization hold.
TXN_5835324,USR_008,8013.09,2000.0,4.0065,Pune,London,-0.1673,AI TRIAGE: Flagged for USR_008. Geo mismatch: initiated at 'London' vs profile city 'Pune' | Spending ceiling breach: $8013.09 exceeds limit $2000.00 | High statistical anomaly index (-0.1673) -> Recommendation: Place authorization hold.
TXN_4013780,USR_002,6956.57,2500.0,2.7826,Bengaluru,Singapore,-0.1673,AI TRIAGE: Flagged for USR_002. Geo mismatch: initiated at 'Singapore' vs profile city 'Bengaluru' | Spending ceiling breach: $6956.57 exceeds limit $2500.00 | High statistical anomaly index (-0.1673) -> Recommendation: Place authorization hold.
TXN_5981390,USR_002,7321.13,2500.0,2.9285,Bengaluru,Dubai,-0.1673,AI TRIAGE: Flagged for USR_002. Geo mismatch: initiated at 'Dubai' vs profile city 'Bengaluru' | Spending ceiling breach: $7321.13 exceeds limit $2500.00 | High statistical anomaly index (-0.1673) -> Recommendation: Place authorization hold.
